# Phase 2 — stitched bridge skeleton: build it, prove the plumbing, and test whether content flows

Plan v2 (`docs/encoder_decoder_bridge_e2e_plan_v2.md`), Phase 2; implementation notes in `docs/stitched-e2e-plan.md`.

```
English source ──► ModernBERT-base (+ LoRA, frozen) ──► bridge: Linear 768→2560 + RMSNorm        [GPU 0]
                                                              │ memory [B, S, 2560] + padding mask
                                                              ▼
instruction ──► Gemma 3 4B IT, 4-bit (+ LoRA, frozen)                                               [GPU 1]
 (no source)      after every 4th layer:  h ← h + tanh(gate) · CrossAttention(norm(h), memory)
                  gates start at 0, so the untouched system *is* Gemma
```

The source reaches the decoder **only through the encoder**: the decoder's prompt never contains the sentence. Cross-attention is attached with forward hooks, so Gemma's own modules are untouched. Data: v3, the three personas kept (Don, Anthony Gonsalves, Jai), with their v4 descriptions.

**Runs so far.**

- **Run 1** (`ce4ec98`): plumbing proven (tests 1, 2, 3, 5, 6). The persona smoke run cut the loss from 10.97 to 3.84, but with **another sentence's memory** the loss was the same (gap 0.002).
- **Run 2** (`d643906`): tests 1–6 pass (test 4 now on logits). The copy test cut the loss from 8.17 to 0.41, but again with no gap (0.009), and every output was the same sentence. The persona run from that bridge: gap −0.0015.

Both runs left the model **two shortcuts** that lower the loss without reading the memory: it could **memorise** the few target sentences (after their first words, teacher forcing makes the rest predictable), and the bias terms in the cross-attention and bridge let it learn a **fixed push** towards "write a line like these", whatever the encoder says.

**Run 3 closes both shortcuts:**

1. **The copy task uses random word sequences** (5–10 words drawn from a fixed word list), freshly generated for every batch. Nothing can be memorised, and after any prefix the next word cannot be guessed: the only way to lower the loss is to read the memory.
2. **No bias terms** in the cross-attention or the bridge, so there is no constant push to fall back on.
3. **More steps** (300 optimizer steps of 4 sequences) at a higher learning rate (3e-4), with **exact copies** and **word accuracy** on 10 held-out sequences reported along the way.

| # | Test | Pass |
|---|---|---|
| 1 | Shapes, devices; hooks fire in every chosen layer | memory `[B, S, 2560]` on GPU 1; one call per block per forward |
| 2 | Gate-0 equivalence | with gates at 0, logits equal plain Gemma's |
| 3 | Padding mask | padding a source does not change its output; removing the mask does |
| 4 | Generation path | cached = uncached; hooks fire at every step; next-token logits change with the cross-attention off and with another sentence's memory |
| 5 | Gradient audit | gradients reach encoder LoRA, bridge, cross-attention, gates and decoder LoRA; none reach frozen base weights |
| 6 | Memory and speed | measured on both GPUs |
| 7 | **Content flow (copy random words)** | on 30 held-out random sequences, loss with another sequence's memory exceeds loss with the right memory by **≥ 0.5** |
| 8 | Persona smoke (warm start from 7) | loss falls; right memory below no memory, and below another sentence's memory by **≥ 0.1** |

In [1]:
# Cell 0 — Hugging Face token (Gemma 3 is gated). Clear it before saving or sharing the notebook.

HF_TOKEN = ""

## Cell 1 — Setup

Needs **two GPUs** (Kaggle: *GPU T4 × 2*). Encoder and bridge on `cuda:0`, decoder and cross-attention on `cuda:1`. Compute in fp32 throughout (Gemma 3 is unstable in fp16; its weights are 4-bit).

In [2]:
# Cell 1 — Packages, settings, versions, pinned revisions

!pip uninstall -y -q torchao                 # Kaggle ships torchao 0.10; peft refuses versions below 0.16
!pip install -q "transformers==5.17.0" "peft==0.21.2" -U bitsandbytes

import json, math, random, re, time, urllib.request

import bitsandbytes
import peft
import torch
import torch.nn as nn
import transformers
from huggingface_hub import HfApi, login

ENCODER, DECODER = "answerdotai/ModernBERT-base", "google/gemma-3-4b-it"
DATA = ("https://raw.githubusercontent.com/skamalj/llms-the-hard-way/hindi-decoder-compare/"
        "project4-hindi-decoder/data/persona_translation_v3/")
PERSONAS = ["don", "anthony_gonsalves", "jai_sholay"]
XATTN_EVERY, XATTN_HEADS = 4, 8
SEED = 42
DTYPE = torch.float32

assert torch.cuda.device_count() >= 2, "this notebook needs two GPUs (Kaggle: GPU T4 x2)"
ENC_DEVICE, DEC_DEVICE = torch.device("cuda:0"), torch.device("cuda:1")
HF_TOKEN = HF_TOKEN.strip() or None
assert HF_TOKEN, "Gemma 3 is gated: paste a Hugging Face token in Cell 0"
login(HF_TOKEN)

api = HfApi()
REV = {r: api.model_info(r, token=HF_TOKEN).sha for r in (ENCODER, DECODER)}
VERSIONS = {"torch": torch.__version__, "transformers": transformers.__version__, "peft": peft.__version__,
            "bitsandbytes": bitsandbytes.__version__,
            "gpus": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())], "revisions": REV}
print(json.dumps(VERSIONS, indent=1))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 96.9 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 46.5 MB/s eta 0:00:00:00:0100:01
{
 "torch": "2.11.0+cu128",
 "transformers": "5.17.0",
 "peft": "0.21.2",
 "bitsandbytes": "0.50.2",
 "gpus": [
  "Tesla T4",
  "Tesla T4"
 ],
 "revisions": {
  "answerdotai/ModernBERT-base": "8949b909ec900327062f0ebf497f51aef5e6f0c8",
  "google/gemma-3-4b-it": "093f9f388b31de276ce2de164bdc2081324b9767"
 }
}


## Cell 2 — Data: two tasks, one record format

Every example is a record of **decoder prompt** (an instruction, never the sentence), **target**, and **source** (what the encoder reads).

- **Copy task** (Test 7): prompt "Repeat the words you are given, in the same order."; source and target = the same **random sequence of 5–10 words**, drawn from the words of the v3 English sources. Training sequences are generated fresh for every batch; 30 held-out sequences (a separate seed) check it.
- **Persona task** (Test 8): prompt = the persona instruction (v4 description); target = the v3 Hinglish line; 120 training and 30 validation pairs.

In [3]:
# Cell 2 — v3 data; copy and persona records

def fetch(name):
    return json.load(urllib.request.urlopen(DATA + name))


RAW = {split: [ex for ex in fetch(f"{split}_v3.json") if ex["persona_id"] in PERSONAS]
       for split in ("train", "validation")}

NAME = {"don": "Don", "anthony_gonsalves": "Anthony Gonsalves", "jai_sholay": "Jai (Sholay)"}
DESC = {
    "don": "Cool, calculating and faintly amused; short, controlled sentences; treats every situation as a move "
           "in a game; a quiet hint of control, never loud.",
    "anthony_gonsalves": "Cheerful, theatrical Bombay street-smart; playful exaggeration and comic comparisons; "
                         "freely mixes English and Hindi.",
    "jai_sholay": "Calm, laconic and practical; dry understatement; loyal; says only what is needed.",
}
PERSONA_INSTRUCTION = ("Translate the English sentence you are given into natural Roman-script Hinglish in the voice "
                       "of {name}. Persona: {desc} Preserve the original meaning; do not add new events or facts.")
COPY_INSTRUCTION = "Repeat the words you are given, in the same order."


def persona_record(ex):
    p = ex["persona_id"]
    return {"id": ex["id"], "source_id": ex["source_id"], "persona": p, "source": ex["source"],
            "target": ex["target"], "prompt": PERSONA_INSTRUCTION.format(name=NAME[p], desc=DESC[p])}


WORDS = sorted({w for rows in RAW.values() for ex in rows for w in re.findall(r"[a-z]+", ex["source"].lower())
                if len(w) > 2})


def random_copy_record(rng, key):
    words = " ".join(rng.choice(WORDS) for _ in range(rng.randint(5, 10)))
    return {"id": f"copy_{key}", "source_id": f"rand{key}", "persona": None, "source": words, "target": words,
            "prompt": COPY_INSTRUCTION}


TRAIN, VAL = [persona_record(ex) for ex in RAW["train"]], [persona_record(ex) for ex in RAW["validation"]]
val_rng = random.Random(SEED + 1000)
COPY_VAL = [random_copy_record(val_rng, f"{k:02d}") for k in range(30)]
assert len({r["source"] for r in COPY_VAL}) == len(COPY_VAL)
assert not any(r["source"] in r["prompt"] for r in TRAIN + VAL + COPY_VAL)
print(f"persona: train {len(TRAIN)}, validation {len(VAL)} | copy: word list {len(WORDS)} words, "
      f"held-out sequences {len(COPY_VAL)}, training sequences generated per batch")
print("encoder input:  ", TRAIN[0]["source"])
print("persona prompt: ", TRAIN[0]["prompt"])
print("persona target: ", TRAIN[0]["target"])
print("copy prompt:    ", COPY_VAL[0]["prompt"], "| source = target:", COPY_VAL[0]["target"])

persona: train 120, validation 30 | copy: word list 210 words, held-out sequences 30, training sequences generated per batch
encoder input:   I don't trust someone who changes their story every time.
persona prompt:  Translate the English sentence you are given into natural Roman-script Hinglish in the voice of Don. Persona: Cool, calculating and faintly amused; short, controlled sentences; treats every situation as a move in a game; a quiet hint of control, never loud. Preserve the original meaning; do not add new events or facts.
persona target:  Har baar nayi story? Trust chahiye toh pehle apni baat ek jaisi rakho.
copy prompt:     Repeat the words you are given, in the same order. | source = target: choose hear avoiding got fixed pretend


## Cell 3 — Load the encoder (GPU 0) and the decoder (GPU 1)

The cell first checks that both GPUs are empty: re-running it in the same kernel would load a second copy of Gemma (4.5 GB) on top of the first and run out of memory later. If models are still loaded, it stops with a message to restart the kernel.

In [4]:
# Cell 3 — ModernBERT-base on GPU 0; Gemma 3 4B IT (4-bit NF4, fp32 compute) on GPU 1

from transformers import AutoConfig, AutoModel, AutoTokenizer, BitsAndBytesConfig, Gemma3ForConditionalGeneration

used = {d: torch.cuda.memory_allocated(d) / 2**30 for d in (ENC_DEVICE, DEC_DEVICE)}
if max(used.values()) > 0.5:
    raise RuntimeError(f"GPU memory already in use ({used[ENC_DEVICE]:.1f} GB / {used[DEC_DEVICE]:.1f} GB): models from an "
                       "earlier run are still loaded. Restart the kernel (Run > Restart & clear outputs) and run all "
                       "cells once, from the top.")

enc_tok = AutoTokenizer.from_pretrained(ENCODER, revision=REV[ENCODER])
enc_cfg = AutoConfig.from_pretrained(ENCODER, revision=REV[ENCODER])
if hasattr(enc_cfg, "reference_compile"):
    enc_cfg.reference_compile = False        # no torch.compile of the embeddings on a T4
encoder = AutoModel.from_pretrained(ENCODER, revision=REV[ENCODER], config=enc_cfg, dtype=DTYPE,
                                    attn_implementation="sdpa").to(ENC_DEVICE).eval()

dec_tok = AutoTokenizer.from_pretrained(DECODER, revision=REV[DECODER], token=HF_TOKEN)
quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=DTYPE)
decoder = Gemma3ForConditionalGeneration.from_pretrained(DECODER, revision=REV[DECODER], dtype=DTYPE, token=HF_TOKEN,
                                                         quantization_config=quant, device_map={"": DEC_DEVICE.index},
                                                         attn_implementation="eager")
TEXT = decoder.config.text_config
D_ENC, D_DEC, N_LAYERS = encoder.config.hidden_size, TEXT.hidden_size, TEXT.num_hidden_layers
LAYERS = next(m for n, m in decoder.named_modules() if n.endswith("language_model.layers"))
assert len(LAYERS) == N_LAYERS
eos = decoder.generation_config.eos_token_id
STOPS = sorted(set(eos if isinstance(eos, list) else [eos]))
PAD = dec_tok.pad_token_id

print(f"encoder width {D_ENC} | decoder width {D_DEC}, {N_LAYERS} layers | stop tokens {STOPS}")
print(f"GPU 0: {torch.cuda.memory_allocated(ENC_DEVICE) / 2**30:.1f} GB | "
      f"GPU 1: {torch.cuda.memory_allocated(DEC_DEVICE) / 2**30:.1f} GB")

config.json:   0%|          | 0.00/1.19k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/20.8k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.13M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  599MB            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

encoder width 768 | decoder width 2560, 34 layers | stop tokens [1, 106]
GPU 0: 0.6 GB | GPU 1: 4.5 GB


## Cell 4 — Bridge, gated cross-attention, hooks, and the batch helpers

- **Bridge** (GPU 0): `Linear(768 → 2560)` **without bias**, then RMSNorm.
- **Gated cross-attention** (GPU 1), one block per chosen layer: pre-norm `nn.MultiheadAttention` **without bias terms** (queries from the decoder, keys and values from the memory, encoder padding masked), added through `tanh(gate)`, gate initialised to 0. Without biases, everything the block adds comes from the memory: there is no constant push to learn.
- **Hooks** on the chosen Gemma layers replace each layer's output `h` with `h + tanh(gate) · CrossAttention(h, memory)`, reading the memory from a small context object set once per batch; the context also counts hook calls.

In [5]:
# Cell 4 — Bridge, gated cross-attention blocks, hooks, encoder and decoder batch helpers

class Bridge(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.proj = nn.Linear(d_in, d_out, bias=False)
        self.norm = nn.RMSNorm(d_out)

    def forward(self, h):
        return self.norm(self.proj(h))


class GatedCrossAttention(nn.Module):
    def __init__(self, d, heads):
        super().__init__()
        self.norm = nn.RMSNorm(d)
        self.attn = nn.MultiheadAttention(d, heads, batch_first=True, bias=False)
        self.gate = nn.Parameter(torch.zeros(1))

    def forward(self, h, memory, pad):
        out, _ = self.attn(self.norm(h), memory, memory, key_padding_mask=pad, need_weights=False)
        return h + torch.tanh(self.gate) * out


class Context:
    enabled, memory, pad, calls = True, None, None, 0


CTX = Context()
XATTN_LAYERS = list(range(XATTN_EVERY - 1, N_LAYERS, XATTN_EVERY))
torch.manual_seed(SEED)
bridge = Bridge(D_ENC, D_DEC).to(ENC_DEVICE)
xattn = nn.ModuleDict({str(i): GatedCrossAttention(D_DEC, XATTN_HEADS) for i in XATTN_LAYERS}).to(DEC_DEVICE)


def make_hook(i):
    block = xattn[str(i)]

    def hook(module, args, output):
        if not CTX.enabled:
            return output
        CTX.calls += 1
        h = output[0] if isinstance(output, tuple) else output
        h = block(h, CTX.memory, CTX.pad)
        return (h, *output[1:]) if isinstance(output, tuple) else h

    return hook


HOOKS = [LAYERS[i].register_forward_hook(make_hook(i)) for i in XATTN_LAYERS]


def encode(sources, train_encoder=False):
    """English sources -> bridge memory on GPU 1 and its padding mask (True = padding)."""
    t = enc_tok(sources, padding=True, truncation=True, max_length=128, return_tensors="pt").to(ENC_DEVICE)
    with torch.set_grad_enabled(train_encoder and torch.is_grad_enabled()):
        h = encoder(input_ids=t["input_ids"], attention_mask=t["attention_mask"]).last_hidden_state
    return bridge(h).to(DEC_DEVICE), (t["attention_mask"] == 0).to(DEC_DEVICE)


def set_memory(memory, pad, enabled=True):
    CTX.memory, CTX.pad, CTX.enabled = memory, pad, enabled


def prompt_text(prompt):
    return dec_tok.apply_chat_template([{"role": "user", "content": prompt}], add_generation_prompt=True,
                                       tokenize=False)


def encode_pair(r):
    msgs = [{"role": "user", "content": r["prompt"]}, {"role": "assistant", "content": r["target"]}]
    full, head = dec_tok.apply_chat_template(msgs, tokenize=False), prompt_text(r["prompt"])
    assert full.startswith(head)
    prompt = dec_tok(head, add_special_tokens=False)["input_ids"]
    reply = dec_tok(full[len(head):], add_special_tokens=False)["input_ids"]
    assert any(t in STOPS for t in reply)
    return torch.tensor(prompt + reply), torch.tensor([-100] * len(prompt) + reply)


def decoder_batch(records):
    pairs = [encode_pair(r) for r in records]
    T = max(len(x) for x, _ in pairs)
    ids = torch.full((len(pairs), T), PAD)
    labels = torch.full((len(pairs), T), -100)
    mask = torch.zeros((len(pairs), T), dtype=torch.long)
    for i, (x, y) in enumerate(pairs):
        ids[i, :len(x)], labels[i, :len(y)], mask[i, :len(x)] = x, y, 1
    return {"input_ids": ids.to(DEC_DEVICE), "attention_mask": mask.to(DEC_DEVICE), "labels": labels.to(DEC_DEVICE),
            "token_type_ids": torch.zeros_like(ids).to(DEC_DEVICE)}


print(f"cross-attention after decoder layers {XATTN_LAYERS} ({len(XATTN_LAYERS)} blocks)")
print(f"bridge {sum(p.numel() for p in bridge.parameters()) / 1e6:.1f} M parameters | "
      f"cross-attention {sum(p.numel() for p in xattn.parameters()) / 1e6:.1f} M")

cross-attention after decoder layers [3, 7, 11, 15, 19, 23, 27, 31] (8 blocks)
bridge 2.0 M parameters | cross-attention 209.7 M


## Cell 5 — LoRA at both ends (created, frozen) and the parameter groups

Encoder LoRA on ModernBERT's `Wqkv`, `Wo`, `Wi`; decoder LoRA on Gemma's attention and MLP projections (language model only). Both are instantiated but stay frozen in Phase 2 (plan v2). Five groups are tracked: encoder LoRA, bridge, cross-attention, gates, decoder LoRA.

In [6]:
# Cell 5 — Encoder and decoder LoRA; parameter groups; trainable = bridge + cross-attention + gates

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

decoder = prepare_model_for_kbit_training(decoder, use_gradient_checkpointing=True,
                                          gradient_checkpointing_kwargs={"use_reentrant": False})
PROJ = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
dec_model = get_peft_model(decoder, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
                                               target_modules=rf".*language_model.*\.({'|'.join(PROJ)})"))
encoder = get_peft_model(encoder, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,
                                             target_modules=["Wqkv", "Wo", "Wi"]))
assert all(LAYERS[i]._forward_hooks for i in XATTN_LAYERS), "the hooks did not survive the LoRA wrapping"

GROUPS = {
    "encoder LoRA": [p for n, p in encoder.named_parameters() if "lora_" in n],
    "bridge": list(bridge.parameters()),
    "cross-attention": [p for n, p in xattn.named_parameters() if not n.endswith("gate")],
    "gates": [b.gate for b in xattn.values()],
    "decoder LoRA": [p for n, p in dec_model.named_parameters() if "lora_" in n],
}


def set_trainable(*names):
    for name, params in GROUPS.items():
        for p in params:
            p.requires_grad_(name in names)


def set_gates(value):
    for b in xattn.values():
        b.gate.data.fill_(value)


SMOKE_GROUPS = ("bridge", "cross-attention", "gates")
set_trainable(*SMOKE_GROUPS)
for name, params in GROUPS.items():
    print(f"  {name:16s} {sum(p.numel() for p in params) / 1e6:8.2f} M parameters  "
          f"{'trainable' if name in SMOKE_GROUPS else 'frozen'}")

  encoder LoRA         3.38 M parameters  frozen
  bridge               1.97 M parameters  trainable
  cross-attention    209.74 M parameters  trainable
  gates                0.00 M parameters  trainable
  decoder LoRA        29.80 M parameters  frozen


## Cell 6 — Tests 1–3: shapes and hooks, gate-0 equivalence, padding mask

The logits are compared on the last 16 positions only (the reply), and freed as soon as they are used: Gemma's vocabulary has 262k entries, so full logits for a small batch take about half a gigabyte each. The cell also prints GPU memory before the tests: with only the models loaded it should be about 0.6 GB on GPU 0 and 5–6 GB on GPU 1; much more means a model was loaded twice (restart the session and run all cells once).

In [7]:
# Cell 6 — Test 1 (shapes, devices, hook calls), Test 2 (gate-0 equivalence), Test 3 (padding mask)

TESTS = {}
dec_model.eval()
encoder.eval()
print(f"before the tests: GPU 0 {torch.cuda.memory_allocated(ENC_DEVICE) / 2**30:.1f} GB, "
      f"GPU 1 {torch.cuda.memory_allocated(DEC_DEVICE) / 2**30:.1f} GB")


TAIL = 16          # compare only the last 16 positions (the reply): full-vocabulary logits are ~0.5 GB per batch


def logits_for(records, sources, enabled=True, no_mask=False):
    dec = decoder_batch(records)
    dec.pop("labels")
    memory, pad = encode(sources)
    set_memory(memory, None if no_mask else pad, enabled)
    CTX.calls = 0
    return dec_model(**dec).logits[:, -TAIL:].clone(), memory, pad


with torch.no_grad():
    # Test 1
    logits, memory, pad = logits_for(VAL[:4], [r["source"] for r in VAL[:4]])
    TESTS["1 shapes, devices, hooks"] = (memory.shape[-1] == D_DEC and memory.device == DEC_DEVICE
                                        and CTX.calls == len(XATTN_LAYERS))
    del logits
    print(f"Test 1: memory {tuple(memory.shape)} on {memory.device}, padding mask {tuple(pad.shape)}, "
          f"hook calls per forward {CTX.calls} (expected {len(XATTN_LAYERS)})")

    # Test 2: with gates at 0 the stitched model is Gemma
    set_gates(0.0)
    plain, _, _ = logits_for(VAL[:4], [r["source"] for r in VAL[:4]], enabled=False)
    stitched, _, _ = logits_for(VAL[:4], [r["source"] for r in VAL[:4]])
    diff = (plain - stitched).abs().max().item()
    del plain, stitched
    TESTS["2 gate-0 equivalence"] = diff <= 1e-5
    print(f"Test 2: max |logit difference| plain vs stitched with gates at 0 = {diff:.2e} (pass <= 1e-5)")

    # Test 3: padding the encoder input must not change the output (gates open so the memory matters)
    set_gates(1.0)
    n_tokens = lambda r: len(enc_tok(r["source"])["input_ids"])
    a, longer = min(VAL, key=n_tokens), max(VAL, key=n_tokens)
    assert n_tokens(longer) > n_tokens(a)
    alone, _, _ = logits_for([a], [a["source"]])
    batched, _, pad = logits_for([a, a], [a["source"], longer["source"]])
    unmasked, _, _ = logits_for([a, a], [a["source"], longer["source"]], no_mask=True)
    d_masked = (alone[0] - batched[0]).abs().max().item()
    d_unmasked = (alone[0] - unmasked[0]).abs().max().item()
    n_pad = int(pad[0].sum())
    del alone, batched, unmasked, memory, pad
    torch.cuda.empty_cache()
    TESTS["3 padding mask"] = d_masked <= 1e-3 and d_unmasked > 100 * max(d_masked, 1e-6)
    print(f"Test 3: source padded with {n_pad} positions | difference with mask {d_masked:.2e} "
          f"(pass <= 1e-3) | without mask {d_unmasked:.2e} (must be clearly larger)")
    set_gates(0.0)

before the tests: GPU 0 0.6 GB, GPU 1 5.4 GB
Test 1: memory (4, 15, 2560) on cuda:1, padding mask (4, 15), hook calls per forward 8 (expected 8)
Test 2: max |logit difference| plain vs stitched with gates at 0 = 0.00e+00 (pass <= 1e-5)
Test 3: source padded with 10 positions | difference with mask 3.11e-04 (pass <= 1e-3) | without mask 6.74e+00 (must be clearly larger)


## Cell 7 — Test 4: the generation path

With the gates open (tanh(1) ≈ 0.76) and untrained cross-attention: cached and uncached generation must give the same tokens; the hooks must fire at every generation step; and the **next-token logits** after the prompt must change when the cross-attention is switched off and when another sentence's memory is given. (Whether the *chosen word* changes is printed for information: untrained cross-attention is usually too weak to flip Gemma's greedy choice, which is what failed in run 1.)

In [8]:
# Cell 7 — Test 4: cached vs uncached generation; hooks at every step; the memory changes the logits

@torch.no_grad()
def generate(r, memory_source=None, enabled=True, use_cache=True, max_new_tokens=24):
    ids = dec_tok(prompt_text(r["prompt"]), return_tensors="pt", add_special_tokens=False).to(DEC_DEVICE)
    memory, pad = encode([memory_source or r["source"]])
    set_memory(memory, pad, enabled)
    CTX.calls = 0
    out = dec_model.generate(**ids, do_sample=False, num_beams=1, max_new_tokens=max_new_tokens,
                             eos_token_id=STOPS, pad_token_id=PAD, use_cache=use_cache,
                             temperature=None, top_p=None, top_k=None)
    return out[0, ids["input_ids"].shape[1]:].tolist(), CTX.calls


@torch.no_grad()
def next_token_logits(r, memory_source=None, enabled=True):
    ids = dec_tok(prompt_text(r["prompt"]), return_tensors="pt", add_special_tokens=False).to(DEC_DEVICE)
    memory, pad = encode([memory_source or r["source"]])
    set_memory(memory, pad, enabled)
    return dec_model(**ids, token_type_ids=torch.zeros_like(ids["input_ids"])).logits[0, -1]


dec_model.eval()
set_gates(1.0)
r, other = VAL[0], next(v for v in VAL if v["source_id"] != VAL[0]["source_id"])
cached, calls = generate(r)
uncached, _ = generate(r, use_cache=False)
right_logits = next_token_logits(r)
d_off = (right_logits - next_token_logits(r, enabled=False)).abs().max().item()
d_other = (right_logits - next_token_logits(r, memory_source=other["source"])).abs().max().item()
off_tokens, _ = generate(r, enabled=False)
other_tokens, _ = generate(r, memory_source=other["source"])
set_gates(0.0)

every_step = calls % len(XATTN_LAYERS) == 0 and calls // len(XATTN_LAYERS) >= len(cached)
TESTS["4 generation path"] = cached == uncached and every_step and d_off > 1e-3 and d_other > 1e-3
print(f"cached == uncached: {cached == uncached} | hook calls {calls} = {len(XATTN_LAYERS)} layers x "
      f"{calls // len(XATTN_LAYERS)} forward passes for {len(cached)} new tokens: {every_step}")
print(f"max |next-token logit change|: cross-attention off {d_off:.3f}, other sentence's memory {d_other:.3f} "
      f"(pass > 1e-3 each)")
print(f"(information) generated words change: off {cached != off_tokens}, other memory {cached != other_tokens}")

[transformers] The following generation flags are not valid and may be ignored: ['cache_implementation']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


cached == uncached: True | hook calls 192 = 8 layers x 24 forward passes for 24 new tokens: True
max |next-token logit change|: cross-attention off 1.596, other sentence's memory 0.473 (pass > 1e-3 each)
(information) generated words change: off False, other memory False


## Cell 8 — Test 5: gradient audit (all five groups)

All five groups trainable for one step, gates at 0.1, one training batch forward and backward with the encoder in the graph. Each group must receive a non-zero gradient; no frozen base weight may receive one. (LoRA's `B` matrices start at zero, so in each LoRA group only `B` receives a gradient on the first step.)

In [9]:
# Cell 8 — Test 5: gradient norms per group; frozen weights must get none

def stitched_loss(records, memory_sources=None, enabled=True, train_encoder=False):
    dec = decoder_batch(records)
    memory, pad = encode(memory_sources or [r["source"] for r in records], train_encoder)
    set_memory(memory, pad, enabled)
    return dec_model(**dec).loss


set_trainable(*GROUPS)
set_gates(0.1)
dec_model.train()
for params in GROUPS.values():
    for p in params:
        p.grad = None
stitched_loss(TRAIN[:2], train_encoder=True).backward()

NORMS = {name: math.sqrt(sum(float(p.grad.norm()) ** 2 for p in params if p.grad is not None))
         for name, params in GROUPS.items()}
tracked = {id(p) for params in GROUPS.values() for p in params}
stray = [n for model in (encoder, dec_model) for n, p in model.named_parameters()
         if id(p) not in tracked and p.grad is not None]
TESTS["5 gradient audit"] = all(v > 0 for v in NORMS.values()) and not stray
for name, v in NORMS.items():
    print(f"  {name:16s} gradient norm {v:.3e}")
print(f"frozen base weights with a gradient: {len(stray)}")

for params in GROUPS.values():
    for p in params:
        p.grad = None
set_gates(0.0)
set_trainable(*SMOKE_GROUPS)
dec_model.eval()

[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


  encoder LoRA     gradient norm 1.193e-01
  bridge           gradient norm 1.434e+00
  cross-attention  gradient norm 3.363e+00
  gates            gradient norm 1.054e-01
  decoder LoRA     gradient norm 5.703e+01
frozen base weights with a gradient: 0


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Gemma3ForConditionalGeneration(
      (model): Gemma3Model(
        (vision_tower): SiglipVisionModel(
          (embeddings): SiglipVisionEmbeddings(
            (patch_embedding): Conv2d(3, 1152, kernel_size=(14, 14), stride=(14, 14), padding=valid)
            (position_embedding): Embedding(4096, 1152)
          )
          (encoder): SiglipEncoder(
            (layers): ModuleList(
              (0-26): 27 x SiglipEncoderLayer(
                (layer_norm1): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
                (self_attn): SiglipAttention(
                  (k_proj): Linear4bit(in_features=1152, out_features=1152, bias=True)
                  (v_proj): Linear4bit(in_features=1152, out_features=1152, bias=True)
                  (q_proj): Linear4bit(in_features=1152, out_features=1152, bias=True)
                  (out_proj): Linear4bit(in_features=1152, out_features=1152, bias=True)
                )
 

## Cell 9 — Training helpers

Two training loops — by steps on fresh random sequences for the copy task, by epochs for the persona task — with the same rules: only the bridge, the cross-attention blocks and their gates train (encoder, decoder and both LoRAs frozen); the gates have their own, larger learning rate so they can open in a short run. After every epoch, three losses on a set of records:

- **right:** each record with its own sentence's memory;
- **other:** each record with the memory of a sentence from a *different* source (a fixed pairing: the next source in id order);
- **none:** cross-attention off — Gemma with the instruction only.

The **gap** `other − right` is what matters: it is how much the loss depends on *which* sentence the encoder read.

In [10]:
# Cell 9 — Losses (right / other / none) and the bridge training loop

MICRO, ACCUM, LR, LR_GATES = 2, 2, 3e-4, 1e-2


def other_sources(records):
    """For each record, the source of the next source id (cyclic) in the same set: always a different sentence."""
    text = {r["source_id"]: r["source"] for r in records}
    ids = sorted(text)
    nxt = {sid: ids[(k + 1) % len(ids)] for k, sid in enumerate(ids)}
    return [text[nxt[r["source_id"]]] for r in records]


@torch.no_grad()
def losses(records, batch=3):
    dec_model.eval()
    others = other_sources(records)
    out = {"right": 0.0, "other": 0.0, "none": 0.0}
    for i in range(0, len(records), batch):
        chunk, oth = records[i:i + batch], others[i:i + batch]
        out["right"] += stitched_loss(chunk).item() * len(chunk)
        out["other"] += stitched_loss(chunk, memory_sources=oth).item() * len(chunk)
        out["none"] += stitched_loss(chunk, enabled=False).item() * len(chunk)
    out = {k: round(v / len(records), 4) for k, v in out.items()}
    out["gap"] = round(out["other"] - out["right"], 4)
    return out


def gate_values():
    return [round(math.tanh(b.gate.item()), 3) for b in xattn.values()]


def make_optimizer():
    return torch.optim.AdamW([{"params": GROUPS["bridge"] + GROUPS["cross-attention"], "lr": LR},
                              {"params": GROUPS["gates"], "lr": LR_GATES}], weight_decay=0.0)


def copy_accuracy(records):
    """Greedy copies of held-out sequences: share copied exactly, and mean share of words right in position."""
    exact, word = 0, []
    for r in records:
        toks, _ = generate(r, max_new_tokens=32)
        out = dec_tok.decode(toks, skip_special_tokens=True).strip().split()
        ref = r["target"].split()
        exact += out == ref
        word.append(sum(a == b for a, b in zip(out, ref)) / len(ref))
    return {"exact": round(exact / len(records), 3), "word": round(sum(word) / len(word), 3)}


def train_copy(steps, eval_every, probe):
    """Train bridge + cross-attention + gates on fresh random sequences; evaluate every `eval_every` steps."""
    opt, rng, t0 = make_optimizer(), random.Random(SEED), time.perf_counter()
    log = [{"step": 0, "train_loss": None, "copy_val": losses(COPY_VAL), "accuracy": copy_accuracy(probe),
            "gates": gate_values()}]
    print(log[-1])
    run = []
    for step in range(1, steps + 1):
        dec_model.train()
        for _ in range(ACCUM):
            batch = [random_copy_record(rng, f"t{step}") for _ in range(MICRO)]
            loss = stitched_loss(batch)
            assert torch.isfinite(loss), "non-finite loss"
            (loss / ACCUM).backward()
            run.append(loss.item())
        opt.step()
        opt.zero_grad()
        if step % eval_every == 0:
            log.append({"step": step, "train_loss": round(sum(run) / len(run), 4), "copy_val": losses(COPY_VAL),
                        "accuracy": copy_accuracy(probe), "gates": gate_values()})
            run = []
            print(log[-1], f"| {time.perf_counter() - t0:.0f}s")
    return log, (time.perf_counter() - t0) / steps


def train_bridge(train, evals, epochs, seed=SEED):
    """Train bridge + cross-attention + gates on `train`; after each epoch, losses on each named set in `evals`."""
    opt = make_optimizer()
    log = [{"epoch": 0, "train_loss": None, **{k: losses(v) for k, v in evals.items()}, "gates": gate_values()}]
    print(log[-1])
    rng, t0, steps = random.Random(seed), time.perf_counter(), 0
    for epoch in range(1, epochs + 1):
        dec_model.train()
        order = rng.sample(train, len(train))
        batches = [order[i:i + MICRO] for i in range(0, len(order), MICRO)]
        run = []
        for n, b in enumerate(batches, 1):
            loss = stitched_loss(b)
            assert torch.isfinite(loss), "non-finite loss"
            (loss / ACCUM).backward()
            run.append(loss.item())
            if n % ACCUM == 0 or n == len(batches):
                opt.step()
                opt.zero_grad()
                steps += 1
        log.append({"epoch": epoch, "train_loss": round(sum(run) / len(run), 4),
                    **{k: losses(v) for k, v in evals.items()}, "gates": gate_values()})
        print(log[-1], f"| {time.perf_counter() - t0:.0f}s")
    return log, (time.perf_counter() - t0) / steps


def show_generations(records, n_tokens=48):
    rows = []
    others = other_sources(records)
    for r, oth in zip(records, others):
        right, _ = generate(r, max_new_tokens=n_tokens)
        wrong, _ = generate(r, memory_source=oth, max_new_tokens=n_tokens)
        rows.append({"id": r["id"], "source": r["source"], "reference": r["target"], "other_source": oth,
                     "right_memory": dec_tok.decode(right, skip_special_tokens=True).strip(),
                     "other_memory": dec_tok.decode(wrong, skip_special_tokens=True).strip()})
    for x in rows:
        print(f"\n{x['id']} | English: {x['source']}")
        print(f"  right memory | {x['right_memory']}")
        print(f"  other memory | {x['other_memory']}   (memory of: {x['other_source']})")
        if x["reference"] != x["source"]:
            print(f"  reference    | {x['reference']}")
    return rows


torch.cuda.reset_peak_memory_stats(ENC_DEVICE)
torch.cuda.reset_peak_memory_stats(DEC_DEVICE)

## Cell 10 — Test 7: content flow — can the bridge make Gemma copy random words?

A fresh bridge (no biases) trains on random word sequences for 300 optimizer steps (4 sequences each, all different). The decoder's prompt is only "Repeat the words you are given, in the same order."; the words reach Gemma only through the encoder. Every 50 steps: the three losses on the 30 held-out sequences, and greedy copies of 10 of them (exact copies; share of words right in position).

- **Pass:** on the held-out sequences, the loss with **another sequence's memory** ends at least **0.5** above the loss with the **right memory**.
- Exact copies and word accuracy show how far the content gets through.

In [11]:
# Cell 10 — Test 7: train the bridge on copying random words; right vs other memory; copy accuracy

COPY_STEPS, EVAL_EVERY = 300, 50
COPY_PROBE = COPY_VAL[:10]
COPY_LOG, COPY_SEC = train_copy(COPY_STEPS, EVAL_EVERY, COPY_PROBE)
last = COPY_LOG[-1]
TESTS["7 content flow (copy random words)"] = last["copy_val"]["gap"] >= 0.5
print(f"\nTest 7: held-out sequences — right {last['copy_val']['right']}, other {last['copy_val']['other']}, "
      f"none {last['copy_val']['none']}, gap {last['copy_val']['gap']} (pass >= 0.5) | exact copies "
      f"{last['accuracy']['exact']:.0%}, words in position {last['accuracy']['word']:.0%}")
COPY_SAMPLES = show_generations(COPY_VAL[:4])

{'step': 0, 'train_loss': None, 'copy_val': {'right': 16.7889, 'other': 16.7889, 'none': 16.7889, 'gap': 0.0}, 'accuracy': {'exact': 0.0, 'word': 0.0}, 'gates': [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]}
{'step': 50, 'train_loss': 7.4618, 'copy_val': {'right': 5.4199, 'other': 5.423, 'none': 16.7889, 'gap': 0.0031}, 'accuracy': {'exact': 0.0, 'word': 0.0}, 'gates': [-0.072, 0.104, 0.156, 0.17, -0.158, 0.203, -0.258, 0.333]} | 172s
{'step': 100, 'train_loss': 5.1269, 'copy_val': {'right': 4.9512, 'other': 4.9601, 'none': 16.7889, 'gap': 0.0089}, 'accuracy': {'exact': 0.0, 'word': 0.0}, 'gates': [-0.068, 0.095, 0.132, 0.147, -0.142, 0.193, -0.271, 0.435]} | 291s
{'step': 150, 'train_loss': 4.8516, 'copy_val': {'right': 4.7585, 'other': 4.7839, 'none': 16.7889, 'gap': 0.0254}, 'accuracy': {'exact': 0.0, 'word': 0.0}, 'gates': [-0.034, 0.089, 0.128, 0.147, -0.128, 0.183, -0.291, 0.471]} | 416s
{'step': 200, 'train_loss': 4.7393, 'copy_val': {'right': 4.7606, 'other': 4.8245, 'none': 16.7889

## Cell 11 — Test 8: persona smoke run, continuing from the copy-trained bridge

The same bridge, cross-attention and gates continue on the persona task (120 training pairs, 3 epochs, fresh optimizer). Run 1 trained the same setup from scratch and reached gap 0.002.

- **Pass:** training loss falls; on validation, the right memory beats no memory and beats another sentence's memory by **≥ 0.1**.

In [12]:
# Cell 11 — Test 8: persona smoke run (warm start); right vs other vs none; sample outputs

PERSONA_EPOCHS = 3
PERSONA_LOG, PERSONA_SEC = train_bridge(TRAIN, {"persona_val": VAL}, PERSONA_EPOCHS)
last = PERSONA_LOG[-1]["persona_val"]
TESTS["8 persona smoke (warm start)"] = (PERSONA_LOG[1]["train_loss"] > PERSONA_LOG[-1]["train_loss"]
                                        and last["right"] < last["none"] and last["gap"] >= 0.1)
print(f"\nTest 8: training loss {PERSONA_LOG[1]['train_loss']} -> {PERSONA_LOG[-1]['train_loss']} | validation: "
      f"right {last['right']}, other {last['other']}, none {last['none']}, gap {last['gap']} (pass >= 0.1)")
first_two = sorted({r["source_id"] for r in VAL})[:2]
PERSONA_SAMPLES = show_generations([r for r in VAL if r["source_id"] in first_two])

{'epoch': 0, 'train_loss': None, 'persona_val': {'right': 10.2966, 'other': 10.3657, 'none': 10.9697, 'gap': 0.0691}, 'gates': [-0.05, 0.087, 0.122, 0.139, -0.131, 0.196, -0.307, 0.52]}
{'epoch': 1, 'train_loss': 4.5183, 'persona_val': {'right': 4.003, 'other': 4.022, 'none': 10.9697, 'gap': 0.019}, 'gates': [-0.03, 0.055, 0.096, 0.104, -0.097, 0.097, -0.191, 0.447]} | 151s
{'epoch': 2, 'train_loss': 3.4028, 'persona_val': {'right': 3.8049, 'other': 3.8034, 'none': 10.9697, 'gap': -0.0015}, 'gates': [-0.027, 0.056, 0.093, 0.1, -0.114, 0.105, -0.191, 0.446]} | 302s
{'epoch': 3, 'train_loss': 2.9671, 'persona_val': {'right': 3.8965, 'other': 3.8992, 'none': 10.9697, 'gap': 0.0027}, 'gates': [-0.04, 0.062, 0.105, 0.103, -0.152, 0.148, -0.254, 0.465]} | 453s

Test 8: training loss 4.5183 -> 2.9671 | validation: right 3.8965, other 3.8992, none 10.9697, gap 0.0027 (pass >= 0.1)

s041_don | English: We should celebrate the progress without ignoring the remaining work.
  right memory | Team k

## Cell 12 — Test 6 and the Phase 2 gate

In [13]:
# Cell 12 — Test 6 (memory, speed); all tests; settings as JSON

PEAK = {"GPU 0": round(torch.cuda.max_memory_allocated(ENC_DEVICE) / 2**30, 2),
        "GPU 1": round(torch.cuda.max_memory_allocated(DEC_DEVICE) / 2**30, 2)}
TESTS["6 memory and speed"] = True
print(f"Test 6: peak memory {PEAK} | seconds per optimizer step (batch {MICRO * ACCUM}): "
      f"copy {COPY_SEC:.2f}, persona {PERSONA_SEC:.2f}")

print("\nPHASE 2 GATE")
for name, ok in sorted(TESTS.items()):
    print(f"  {'PASS' if ok else 'FAIL'}  Test {name}")
print("\nall pass" if all(TESTS.values()) else "\nnot all tests pass")

SETTINGS = {
    "versions": VERSIONS,
    "encoder": {"model": ENCODER, "width": D_ENC, "lora": {"r": 16, "alpha": 32, "targets": ["Wqkv", "Wo", "Wi"]}},
    "decoder": {"model": DECODER, "width": D_DEC, "layers": N_LAYERS, "precision": "4-bit NF4, fp32 compute",
                "lora": {"r": 16, "alpha": 32, "targets": PROJ}},
    "bridge": "Linear + RMSNorm",
    "cross_attention": {"after_layers": XATTN_LAYERS, "heads": XATTN_HEADS, "gate": "tanh, initialised to 0"},
    "devices": {"encoder + bridge": str(ENC_DEVICE), "decoder + cross-attention": str(DEC_DEVICE)},
    "training": {"trainable": list(SMOKE_GROUPS), "micro_batch": MICRO, "accumulation": ACCUM, "lr": LR,
                 "lr_gates": LR_GATES, "copy_steps": COPY_STEPS, "copy_words": len(WORDS), "persona_epochs": PERSONA_EPOCHS,
                 "biases": "none in bridge and cross-attention",
                 "copy_log": COPY_LOG, "persona_log": PERSONA_LOG, "peak_memory_gb": PEAK,
                 "seconds_per_step": {"copy": round(COPY_SEC, 2), "persona": round(PERSONA_SEC, 2)}},
    "gradient_norms": NORMS,
    "tests": TESTS,
    "copy_samples": COPY_SAMPLES,
    "persona_samples": PERSONA_SAMPLES,
}
print("\n----- SETTINGS -----")
print(json.dumps(SETTINGS, ensure_ascii=False, indent=1, default=str))

Test 6: peak memory {'GPU 0': 0.64, 'GPU 1': 8.63} | seconds per optimizer step (batch 4): copy 2.67, persona 5.04

PHASE 2 GATE
  PASS  Test 1 shapes, devices, hooks
  PASS  Test 2 gate-0 equivalence
  PASS  Test 3 padding mask
  PASS  Test 4 generation path
  PASS  Test 5 gradient audit
  PASS  Test 6 memory and speed
  FAIL  Test 7 content flow (copy random words)
  FAIL  Test 8 persona smoke (warm start)

not all tests pass

----- SETTINGS -----
{
 "versions": {
  "torch": "2.11.0+cu128",
  "transformers": "5.17.0",
  "peft": "0.21.2",
  "bitsandbytes": "0.50.2",
  "gpus": [
   "Tesla T4",
   "Tesla T4"
  ],
  "revisions": {
   "answerdotai/ModernBERT-base": "8949b909ec900327062f0ebf497f51aef5e6f0c8",
   "google/gemma-3-4b-it": "093f9f388b31de276ce2de164bdc2081324b9767"
  }
 },
 "encoder": {
  "model": "answerdotai/ModernBERT-base",
  "width": 768,
  "lora": {
   "r": 16,
   "alpha": 32,
   "targets": [
    "Wqkv",
    "Wo",
    "Wi"
   ]
  }
 },
 "decoder": {
  "model": "google/ge

---
## Next

Save the notebook (clear the token in Cell 0 first).

- **Test 7 passes:** the bridge can carry content. If Test 8 also passes, Phase 3 trains encoder LoRA, bridge, cross-attention and decoder LoRA together on the v4 data, starting with this random-word copy warm-up. If Test 8 fails, the persona task needs Phase 3's full training (both LoRAs on) after the warm-up.
- **Test 7 fails:** even with nothing to memorise and no constant push, the bridge does not carry the words. The design changes before any further training: the size of the cross-attention's contribution relative to Gemma's hidden states, cross-attention in more layers, or a longer warm-up; and plan proposal P1 (a public English → Hinglish corpus) becomes the next step.